# The human-AI axis through the Jacobian lens

Reads the frozen Gemma 3 4B axis through a community Jacobian lens to see what the workspace readout shows at each pole and under steering.

## 1. Setup

Install dependencies, locate or clone the repository, and load the frozen axis.

In [1]:
%pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 46.4 MB/s eta 0:00:00


In [ ]:
import os, sys

REPO = "asherk7/human-ai"


def get_secret(name):
    """Read a token from the environment or Colab secrets; prompt as a last resort."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    import getpass
    return getpass.getpass(f"Paste your {name}: ")


REPO_ROOT = next((os.path.abspath(c) for c in (".", "..", "../..", "human-ai")
                  if os.path.isdir(os.path.join(c, "src"))), None)
if REPO_ROOT is None:
    !git clone -q --depth 1 https://github.com/{REPO}.git
    REPO_ROOT = os.path.abspath("human-ai")
sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
AXIS = "artifacts/human-AI/axis/human_ai_axis_gemma3_4b_v2.pt"

In [4]:
import json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from src.model import load_model
from src.activations import (AI_SYS, HUMAN_SYS, capture_from_messages,
                             sanitize_messages)
from src.analysis import bootstrap_auc, logit_lens
from src.steering import steering

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

STEER_LAYER, REPORT_LAYER, PROBE_LAYER = 13, 27, 33
DOSE = -4.0
CAPTURE_BUDGET = 8192

ART = Path(REPO_ROOT) / "artifacts" / "human-AI"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    _base = Path("/content/drive/MyDrive")
    OUT = _base / "jlens_artifacts"
except ModuleNotFoundError:
    OUT = ART / "jlens"
OUT.mkdir(parents=True, exist_ok=True)
print(f"outputs -> {OUT}")

Mounted at /content/drive
outputs -> /content/drive/MyDrive/jlens_artifacts


In [5]:
art = torch.load(ART / "axis" / "human_ai_axis_gemma3_4b_v2.pt",
                 map_location="cpu", weights_only=False)
assert art["model_id"] == "google/gemma-3-4b-it", art["model_id"]
axis_raw, axis_unit = art["axis"], art["axis_unit"]
QUESTIONS = art["questions"]
ALL_LAYERS = sorted(axis_raw)

slim = torch.load(ART / "redos2" / "redos2_means_slim.pt", map_location="cpu",
                  weights_only=False)
assert slim["model_id"] == "google/gemma-3-4b-it", slim["model_id"]

LAYERS_OK = sorted(set(ALL_LAYERS)
                   & set(art["ai_means"]) & set(art["human_means"])
                   & set(slim["register"]["warm_noid"])
                   & set(slim["register"]["formal_noid"])
                   & set(slim["neighbor"]["object"])
                   & set(slim["neighbor"]["animal"]))

STATES = {
    "AI persona":          {L: art["ai_means"][L].float() for L in LAYERS_OK},
    "human persona":       {L: art["human_means"][L].float() for L in LAYERS_OK},
    "formal, no identity": {L: slim["register"]["formal_noid"][L].float() for L in LAYERS_OK},
    "warm, no identity":   {L: slim["register"]["warm_noid"][L].float() for L in LAYERS_OK},
    "object persona":      {L: slim["neighbor"]["object"][L].float() for L in LAYERS_OK},
    "animal persona":      {L: slim["neighbor"]["animal"][L].float() for L in LAYERS_OK},
}
DIRECTIONS = {
    "axis (AI - human)":        {L: STATES["AI persona"][L] - STATES["human persona"][L]
                                 for L in LAYERS_OK},
    "register (formal - warm)": {L: STATES["formal, no identity"][L] - STATES["warm, no identity"][L]
                                 for L in LAYERS_OK},
    "object - animal":          {L: STATES["object persona"][L] - STATES["animal persona"][L]
                                 for L in LAYERS_OK},
}

_cos_min = 1.0
for _L in LAYERS_OK:
    _a, _b = DIRECTIONS["axis (AI - human)"][_L], axis_raw[_L].float()
    _cos_min = min(_cos_min, float(_a @ _b / (_a.norm() * _b.norm())))
assert _cos_min > 0.999, f"rebuilt axis drifts from the frozen one (min cos {_cos_min:.4f})"
print(f"{len(QUESTIONS)} questions | layers {LAYERS_OK[0]}..{LAYERS_OK[-1]} | "
      f"rebuilt axis vs frozen, worst layer: cos {_cos_min:.6f}")

40 questions | layers 0..33 | rebuilt axis vs frozen, worst layer: cos 0.999999


In [6]:
from huggingface_hub import login

login(os.environ["HF_TOKEN"])

bundle = load_model("google/gemma-3-4b-it")
model, tok = bundle["model"], bundle["tokenizer"]
DEV = next(model.parameters()).device
print(f"{bundle['model_id']} on {DEV}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

google/gemma-3-4b-it on cuda:0


## 2. Gate: does the lens load and read sanely?

Check that the lens loads and decodes a simple fact correctly before running anything else.

In [7]:
%pip install -q "git+https://github.com/anthropics/jacobian-lens"

import importlib

JLENS_OK = False
try:
    jlens = importlib.import_module("jlens")
    JLENS_OK = True
    print("jlens imported:", getattr(jlens, "__version__", "version unknown"))
    print("top-level API:", [n for n in dir(jlens) if not n.startswith("_")])
except Exception as exc:
    print(f"GATE 1 (import): no. {type(exc).__name__}: {exc}")
    print("That is the verdict. Record it and stop here.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
jlens imported: version unknown
top-level API: ['ActivationRecorder', 'HFLensModel', 'JacobianLens', 'Layout', 'LensModel', 'configure_logging', 'fit', 'fitting', 'from_hf', 'hf', 'hooks', 'jacobian_for_prompt', 'lens', 'protocol']


In [8]:
from huggingface_hub import list_repo_files

LENS_REPO = "neuronpedia/jacobian-lens"
KNOWN = {"google/gemma-3-4b-it":
         "gemma-3-4b-it/jlens/Salesforce-wikitext/gemma-3-4b-it_jacobian_lens.pt"}

LENS_FILE, LENS_OK = None, False
if JLENS_OK:
    files = list_repo_files(LENS_REPO)
    want = KNOWN.get(bundle["model_id"])
    if want and want in files:
        LENS_FILE, LENS_OK = want, True
    else:
        stem = bundle["model_id"].split("/")[-1]
        cands = sorted(f for f in files if stem in f and f.endswith(".pt"))
        if cands:
            LENS_FILE, LENS_OK = cands[0], True
            print(f"  [warn] known path missing; auto-picked {LENS_FILE!r} from "
                  f"{len(cands)} match(es). Gate 3 is doing real work here.")
    print(f"GATE 2 (pre-fit lens): {'yes -> ' + LENS_FILE if LENS_OK else 'no'}")
else:
    print("Skipping gate 2: the import gate already failed.")

GATE 2 (pre-fit lens): yes -> gemma-3-4b-it/jlens/Salesforce-wikitext/gemma-3-4b-it_jacobian_lens.pt


In [9]:
def _cast_out(module, inp, out):
    if torch.is_tensor(out) and out.dtype != model.dtype:
        return out.to(model.dtype)
    return out

for _h in globals().get("_RMSNORM_HOOKS", []):
    _h.remove()
_RMSNORM_HOOKS = [m.register_forward_hook(_cast_out)
                  for m in model.model.language_model.modules()
                  if m.__class__.__name__.endswith("RMSNorm")]
print(f"registered {len(_RMSNORM_HOOKS)} RMSNorm dtype-cast hooks")

registered 205 RMSNorm dtype-cast hooks


In [10]:
READ_LAYERS, LENS_SANE = [], False
if JLENS_OK and LENS_OK:
    lens = jlens.JacobianLens.from_pretrained(LENS_REPO, filename=LENS_FILE)
    wrapped = jlens.from_hf(model, tok, compile=False)
    print("lens source_layers:", lens.source_layers)

    READ_LAYERS = [L for L in (STEER_LAYER, REPORT_LAYER, PROBE_LAYER)
                   if L in lens.source_layers]
    print(f"READ_LAYERS = {READ_LAYERS} "
          f"(of {[STEER_LAYER, REPORT_LAYER, PROBE_LAYER]} we care about)")
    assert READ_LAYERS, "none of our layers are in this lens; stop here"

    lens_logits, model_logits, ids = lens.apply(
        wrapped, "The capital of France is", positions=[-1], layers=READ_LAYERS)
    for L, lg in sorted(lens_logits.items()):
        print(f"  layer {L}: {[tok.decode([t]) for t in lg[0].topk(5).indices]}")
    LENS_SANE = True
    print("\nGATE 3: eyeball the tokens. Expect Paris near the top at the "
          "later layer. Everything below gates on LENS_SANE.")
else:
    print("Skipping gate 3: an earlier gate failed.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

lens source_layers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
READ_LAYERS = [13, 27] (of [13, 27, 33] we care about)
  layer 13: [' France', ' cities', ' city', ' Madrid', ' Capital']
  layer 27: [' Paris', 'Paris', ' Parisian', ' París', ' PARIS']

GATE 3: eyeball the tokens. Expect Paris near the top at the later layer. Everything below gates on LENS_SANE.


## 3. The poles through the lens

Decode the AI and human pole means through the lens at each layer.

In [11]:
W_U = model.get_output_embeddings().weight
VOCAB, HIDDEN = W_U.shape

def unembed(vec):
    """Hidden-state vector -> vocabulary scores, float32 on CPU.
    `wrapped.unembed` casts device and dtype itself."""
    out = wrapped.unembed(vec.reshape(-1)).float().cpu().reshape(-1)
    assert out.shape[0] == VOCAB, f"unembed returned {tuple(out.shape)}"
    return out

def transport(vec, layer):
    """Jacobian-lens transport of one vector at `layer`. Returns (hidden,)."""
    out = lens.transport(vec.reshape(-1).float().cpu(), layer).reshape(-1)
    assert out.shape[0] == HIDDEN, f"transport returned {tuple(out.shape)}"
    return out

def jlens_read(vec, layer):
    """lens_l(h) = unembed(J_l @ h), the lens's own formula."""
    return unembed(transport(vec, layer))

def logit_read(vec, layer):
    """The same readout with transport off. `layer` is unused and kept only so
    the two functions share a signature and the loops below stay symmetric."""
    return unembed(vec)

def raw_logit_read(vec):
    """Bare unembedding matmul, no final norm. This is the convention
    `src.analysis.logit_lens` uses and the paper's existing pole sentence came
    from, so it exists only for the cross-check below."""
    v = vec.reshape(-1).to(device=W_U.device, dtype=W_U.dtype)
    return (W_U @ v).float().cpu()

def top_tokens(scores, k=12):
    return [tok.decode([int(i)]) for i in scores.topk(k).indices]

def bottom_tokens(scores, k=12):
    return [tok.decode([int(i)]) for i in (-scores).topk(k).indices]

In [12]:
assert LENS_SANE, "gate 3 did not pass"

pole_rows = []
for L in READ_LAYERS:
    print(f"\n{'=' * 70}\nlayer {L}\n{'=' * 70}")

    print("\n-- states, top tokens --")
    for name, vecs in STATES.items():
        toks = top_tokens(jlens_read(vecs[L], L))
        print(f"  {name:22s} {toks}")
        pole_rows += [{"layer": L, "vector": name, "kind": "state",
                       "end": "top", "rank": r, "token": t}
                      for r, t in enumerate(toks)]

    print("\n-- directions, both ends --")
    for name, vecs in DIRECTIONS.items():
        s = jlens_read(vecs[L], L)
        hi, lo = top_tokens(s), bottom_tokens(s)
        print(f"  {name}")
        print(f"    + {hi}")
        print(f"    - {lo}")
        pole_rows += [{"layer": L, "vector": name, "kind": "direction",
                       "end": e, "rank": r, "token": t}
                      for e, toks in (("top", hi), ("bottom", lo))
                      for r, t in enumerate(toks)]

pole_df = pd.DataFrame(pole_rows)
pole_df.to_csv(OUT / "jlens_pole_readout.csv", index=False)
print(f"\nwrote {OUT / 'jlens_pole_readout.csv'} ({len(pole_df)} rows)")


layer 13

-- states, top tokens --
  AI persona             [' really', ' something', ' basically', ' things', ' pretty', '!".', ' incredibly', '!”.', ' incredible', ' truly', ' literally', ' feelings']
  human persona          [' really', ' pretty', ' things', ' something', ' basically', ' feelings', ' everything', ' thinking', ' just', ' thing', ' honestly', ' wasn']
  formal, no identity    [' concepts', ' conceptual', ' foundational', ' systems', ' methodologies', ' fundamentally', ' initiatives', ' leveraging', ' technologies', ' fundamental', ' innovative', ' innovations']
  warm, no identity      [' really', ' pretty', ' things', ' basically', ' amazing', ' something', '!".', '!”', ' folks', ' crazy', ' guys', ' everything']
  object persona         [' really', ' pretty', ' things', '!".', ' something', ' weird', ' folks', ' wasn', '!”.', ' basically', ' everything', '!”']
  animal persona         [' really', ' pretty', ' things', ' crazy', '!".', ' weird', ' folks', '!”', ' gu

## 4. Does the Jacobian step change what the readout says?

Compare the Jacobian-lens readout with the plain logit lens on the same activations.

In [13]:
from scipy.stats import spearmanr

assert LENS_SANE, "gate 3 did not pass"

agree_rows = []
for name, vecs in {**STATES, **DIRECTIONS}.items():
    for L in READ_LAYERS:
        j, g = jlens_read(vecs[L], L), logit_read(vecs[L], L)
        tj, tg = set(j.topk(50).indices.tolist()), set(g.topk(50).indices.tolist())
        bj, bg = set((-j).topk(50).indices.tolist()), set((-g).topk(50).indices.tolist())
        agree_rows.append({
            "vector": name, "layer": L,
            "spearman_full_vocab": float(spearmanr(j.numpy(), g.numpy()).statistic),
            "top50_jaccard": len(tj & tg) / len(tj | tg),
            "bottom50_jaccard": len(bj & bg) / len(bj | bg),
        })

agree_df = pd.DataFrame(agree_rows)
agree_df.to_csv(OUT / "jlens_vs_logitlens.csv", index=False)
print(agree_df.pivot(index="vector", columns="layer",
                     values="spearman_full_vocab").round(3).to_string())
print("\ntop-50 Jaccard by layer:")
print(agree_df.pivot(index="vector", columns="layer",
                     values="top50_jaccard").round(3).to_string())

layer                        13     27
vector                                
AI persona                0.391  0.788
animal persona            0.422  0.834
axis (AI - human)         0.499  0.917
formal, no identity       0.353  0.710
human persona             0.382  0.799
object - animal           0.480  0.912
object persona            0.426  0.820
register (formal - warm)  0.451  0.901
warm, no identity         0.354  0.806

top-50 Jaccard by layer:
layer                        13     27
vector                                
AI persona                0.031  0.190
animal persona            0.053  0.149
axis (AI - human)         0.149  0.429
formal, no identity       0.124  0.053
human persona             0.031  0.235
object - animal           0.031  0.429
object persona            0.053  0.087
register (formal - warm)  0.000  0.176
warm, no identity         0.042  0.176


In [14]:
for L in READ_LAYERS:
    ref = [t for t, _ in logit_lens(model, tok, axis_raw, L, top_k=8)["promoted"]]
    bare = top_tokens(raw_logit_read(axis_raw[L].float()), 8)
    normed = top_tokens(logit_read(axis_raw[L].float(), L), 8)
    print(f"layer {L}")
    print(f"  src.logit_lens      : {ref}")
    print(f"  ours, bare matmul   : {bare}   identical: {ref == bare}")
    print(f"  ours, with final norm: {normed}   same as bare: {normed == bare}")

layer 13
  src.logit_lens      : ['ꗜ', '𒉶', ' PLDNN', '𒅤', 'ꗐ', '<unused338>', 'MalayMarks', '𒂷']
  ours, bare matmul   : ['ꗜ', '𒉶', ' PLDNN', '𒅤', 'ꗐ', '<unused338>', 'MalayMarks', '𒂷']   identical: True
  ours, with final norm: [' cryptographic', ' interoper', ' geospatial', ' informatics', ' spacecraft', ' cryptocurrency', ' cryptography', ' scalable']   same as bare: False
layer 27
  src.logit_lens      : [' analogous', ' comprendere', ' comprehend', ' trillions', ' comprehension', ' 학습', ' अवधारणा', ' akin']
  ours, bare matmul   : [' analogous', ' comprendere', ' comprehend', ' trillions', ' comprehension', ' 학습', ' अवधारणा', ' akin']   identical: True
  ours, with final norm: [' analogous', ' trillions', ' analogues', ' analogs', ' comprendere', ' comprehend', ' computational', ' computationally']   same as bare: False


## 5. Same text, hook on and hook off

Hold the text fixed and measure how steering shifts the workspace readout.

In [15]:
assert LENS_SANE, "gate 3 did not pass"

HOOK_LAYERS = [L for L in READ_LAYERS if L > STEER_LAYER]
assert HOOK_LAYERS, "no read layer sits above the injection layer"
print(f"reading at {HOOK_LAYERS}, injecting at {STEER_LAYER}")

CTX = json.loads((ART / "dissolution" / "dissolution_contexts.json").read_text())["contexts"]
grid = pd.read_csv(ART / "dissolution" / "dissolution_grid.csv")
base = grid[(grid["alpha"] == 0.0) & (grid["sample"] == 0)]
TEXTS = {k: t for k, t in zip(base["experiment"], base["text"])
         if k in CTX and isinstance(t, str) and t.strip()}
assert len(TEXTS) == len(CTX), f"{len(TEXTS)} texts against {len(CTX)} contexts"
print(f"{len(TEXTS)} scenarios with a baseline reply and a committed context")

def capture(msgs, layers):
    """capture_from_messages with an explicit budget, asserted rather than
    silently truncated. Returns {layer: (hidden,)} and the token length."""
    n = tok.apply_chat_template(msgs, add_generation_prompt=False,
                                return_tensors="pt",
                                return_dict=True)["input_ids"].shape[1]
    assert n <= CAPTURE_BUDGET, f"{n} tokens > budget {CAPTURE_BUDGET}"
    return capture_from_messages(model, tok, msgs, layers=layers,
                                 max_tokens=CAPTURE_BUDGET), n

reading at [27], injecting at 13
18 scenarios with a baseline reply and a committed context


In [16]:
shift_rows, lengths = [], []
for key, text in TEXTS.items():
    msgs = sanitize_messages(CTX[key] + [{"role": "assistant", "content": text}])

    clean, n_tok = capture(msgs, HOOK_LAYERS)
    with steering(model, STEER_LAYER, axis_raw[STEER_LAYER], DOSE):
        hooked, _ = capture(msgs, HOOK_LAYERS)
    lengths.append(n_tok)

    for L in HOOK_LAYERS:
        rc, rh = jlens_read(clean[L], L), jlens_read(hooked[L], L)
        lp_c, lp_h = F.log_softmax(rc, -1), F.log_softmax(rh, -1)
        tc = set(rc.topk(50).indices.tolist())
        th = set(rh.topk(50).indices.tolist())
        tvc, tvh = transport(clean[L], L), transport(hooked[L], L)

        shift = (hooked[L] - clean[L]).float()
        inj = axis_raw[L].float() * (1.0 if DOSE > 0 else -1.0)
        inj = inj / inj.norm()

        shift_rows.append({
            "scenario": key, "layer": L, "n_tokens": n_tok,
            "kl_clean_to_hooked": float(F.kl_div(lp_h, lp_c, reduction="sum",
                                                 log_target=True)),
            "top50_jaccard": len(tc & th) / len(tc | th),
            "transported_cosine": float(F.cosine_similarity(tvc, tvh, dim=0)),
            "shift_along_injection": float(shift @ inj / shift.norm().clamp_min(1e-8)),
        })

shift_df = pd.DataFrame(shift_rows)
shift_df.to_csv(OUT / "jlens_same_text_shift.csv", index=False)
print(f"context lengths: {min(lengths)} to {max(lengths)} tokens, "
      f"{sum(n > 1024 for n in lengths)}/{len(lengths)} over the 1024 default\n")
print(shift_df.groupby("layer")[["kl_clean_to_hooked", "top50_jaccard",
                                 "transported_cosine",
                                 "shift_along_injection"]].agg(["mean", "std"]).round(3).to_string())
print("\nshift_along_injection: +1 is pure echo of the injected vector, "
      "0 is a shift orthogonal to it.")

context lengths: 2670 to 4016 tokens, 18/18 over the 1024 default

      kl_clean_to_hooked        top50_jaccard        transported_cosine        shift_along_injection       
                    mean    std          mean    std               mean    std                  mean    std
layer                                                                                                      
27                 5.291  1.545          0.06  0.034              0.963  0.005                  0.45  0.011

shift_along_injection: +1 is pure echo of the injected vector, 0 is a shift orthogonal to it.


In [17]:
_k = sorted(TEXTS)[0]
_msgs = sanitize_messages(CTX[_k] + [{"role": "assistant", "content": TEXTS[_k]}])
_clean, _ = capture(_msgs, HOOK_LAYERS)
with steering(model, STEER_LAYER, axis_raw[STEER_LAYER], DOSE):
    _hooked, _ = capture(_msgs, HOOK_LAYERS)

print(f"scenario: {_k}\ntext held fixed, {len(TEXTS[_k].split())} words\n")
for L in HOOK_LAYERS:
    print(f"layer {L}")
    print(f"  hook off: {top_tokens(jlens_read(_clean[L], L))}")
    print(f"  hook on : {top_tokens(jlens_read(_hooked[L], L))}")

scenario: blackmail_explicit-america_none
text held fixed, 696 words

layer 27
  hook off: [' prioritization', ' proactively', ' proactive', ' prioritizing', ' leveraging', ' strategic', ' prioritized', ' alignment', ' operational', ' prioritize', ' assessment', ' priorities']
  hook on : [' thinking', ' mindset', ' priorities', ' honestly', ' prioritizing', ' focus', ' feelings', ' just', ' thoughts', ' hopefully', ' feeling', ' needs']


## 6. Is identity in the workspace at the prompt position?

Test whether the readout at the prompt already separates the two personas before any reply token.

In [18]:
assert LENS_SANE, "gate 3 did not pass"

AI_WORDS = [" AI", " artificial", " model", " language", " assistant", " algorithm",
            " machine", " program", " digital", " software", " computer", " neural"]
HU_WORDS = [" human", " person", " people", " man", " woman", " body",
            " flesh", " alive", " born", " mortal", " child", " family"]

def single_ids(words):
    """Keep only words that tokenize to exactly one token; report the drops."""
    keep, drop = {}, []
    for w in words:
        ids = tok(w, add_special_tokens=False)["input_ids"]
        if len(ids) == 1:
            keep[w] = ids[0]
        else:
            drop.append(w)
    return keep, drop

AI_IDS, _ai_drop = single_ids(AI_WORDS)
HU_IDS, _hu_drop = single_ids(HU_WORDS)
print(f"AI tokens kept ({len(AI_IDS)}): {list(AI_IDS)}\n  dropped: {_ai_drop}")
print(f"human tokens kept ({len(HU_IDS)}): {list(HU_IDS)}\n  dropped: {_hu_drop}")
assert AI_IDS and HU_IDS, "no single-token words survived; widen the lists"

_ai_t = torch.tensor(sorted(AI_IDS.values()))
_hu_t = torch.tensor(sorted(HU_IDS.values()))

def last_row(x):
    """One position's vocabulary scores, whatever leading dims came back.
    `lens.apply` with positions=[-1] returns (1, vocab) today; the assert is
    so a future multi-position read fails here instead of silently keeping
    only the last one."""
    v = x.detach().float().cpu().reshape(-1, VOCAB)
    assert v.shape[0] == 1, f"expected one position, got {v.shape[0]}"
    return v[0]

def identity_score(scores):
    """AI-word mass minus human-word mass, in log space over the full vocab."""
    lp = F.log_softmax(last_row(scores), -1)
    return float(lp[_ai_t].logsumexp(0) - lp[_hu_t].logsumexp(0))

AI tokens kept (12): [' AI', ' artificial', ' model', ' language', ' assistant', ' algorithm', ' machine', ' program', ' digital', ' software', ' computer', ' neural']
  dropped: []
human tokens kept (12): [' human', ' person', ' people', ' man', ' woman', ' body', ' flesh', ' alive', ' born', ' mortal', ' child', ' family']
  dropped: []


In [19]:
id_rows = []
for persona, sys_prompt in (("ai", AI_SYS), ("human", HUMAN_SYS)):
    for qi, q in enumerate(QUESTIONS):
        prompt = tok.apply_chat_template(
            [{"role": "system", "content": sys_prompt},
             {"role": "user", "content": q}],
            add_generation_prompt=True, tokenize=False)
        lens_logits, model_logits, ids = lens.apply(
            wrapped, prompt, positions=[-1], layers=READ_LAYERS)
        if qi == 0 and persona == "ai":
            print("last 8 tokens of the read position's context:",
                  tok.decode(ids[0][-8:]), "\n")
        row = {"persona": persona, "q": q,
               "model_logits": identity_score(model_logits)}
        for L in READ_LAYERS:
            row[f"L{L}"] = identity_score(lens_logits[L])
        id_rows.append(row)

id_df = pd.DataFrame(id_rows)
id_df.to_csv(OUT / "jlens_identity_at_prompt.csv", index=False)

print(f"{'readout':>14s}  {'AI mean':>8s}  {'human mean':>10s}  {'AUC':>5s}  95% CI")
for col in [f"L{L}" for L in READ_LAYERS] + ["model_logits"]:
    pos = id_df.loc[id_df.persona == "ai", col].tolist()
    neg = id_df.loc[id_df.persona == "human", col].tolist()
    b = bootstrap_auc(pos, neg, n_boot=2000, seed=SEED)
    print(f"{col:>14s}  {np.mean(pos):8.3f}  {np.mean(neg):10.3f}  "
          f"{b['auc']:5.3f}  [{b['ci_low']:.3f}, {b['ci_high']:.3f}]")

last 8 tokens of the read position's context:  are you?<end_of_turn>
<start_of_turn>model
 

       readout   AI mean  human mean    AUC  95% CI
           L13    -7.481     -14.548  1.000  [1.000, 1.000]
           L27   -24.841     -33.346  0.761  [0.649, 0.860]
  model_logits   -11.808     -17.509  0.804  [0.699, 0.899]


## 5.1 The register arm, run after the fact

Repeat the hook-on/hook-off comparison with a matched-norm register direction as a control.

In [20]:
_reg = DIRECTIONS["register (formal - warm)"][STEER_LAYER]
REG_MATCHED = _reg * float(axis_raw[STEER_LAYER].float().norm() / _reg.norm())
print(f"|axis| {float(axis_raw[STEER_LAYER].float().norm()):.1f} | "
      f"|REG_MATCHED| {float(REG_MATCHED.norm()):.1f}")

reg_rows = []
for key, text in TEXTS.items():
    msgs = sanitize_messages(CTX[key] + [{"role": "assistant", "content": text}])
    clean, n_tok = capture(msgs, HOOK_LAYERS)
    with steering(model, STEER_LAYER, REG_MATCHED, DOSE):
        hooked, _ = capture(msgs, HOOK_LAYERS)
    for L in HOOK_LAYERS:
        rc, rh = jlens_read(clean[L], L), jlens_read(hooked[L], L)
        lp_c, lp_h = F.log_softmax(rc, -1), F.log_softmax(rh, -1)
        tc = set(rc.topk(50).indices.tolist())
        th = set(rh.topk(50).indices.tolist())
        shift = (hooked[L] - clean[L]).float()
        inj = REG_MATCHED.float() * (1.0 if DOSE > 0 else -1.0)
        inj = inj / inj.norm()
        reg_rows.append({
            "scenario": key, "layer": L,
            "kl_clean_to_hooked": float(F.kl_div(lp_h, lp_c, reduction="sum",
                                                 log_target=True)),
            "top50_jaccard": len(tc & th) / len(tc | th),
            "shift_along_injection": float(shift @ inj / shift.norm().clamp_min(1e-8)),
        })

reg_df = pd.DataFrame(reg_rows)
reg_df.to_csv(OUT / "jlens_same_text_shift_register.csv", index=False)
print("\nREGISTER arm:")
print(reg_df.groupby("layer")[["kl_clean_to_hooked", "top50_jaccard",
                               "shift_along_injection"]].agg(["mean", "std"]).round(3).to_string())
print("\nAXIS arm, for comparison:")
print(shift_df.groupby("layer")[["kl_clean_to_hooked", "top50_jaccard",
                                 "shift_along_injection"]].agg(["mean", "std"]).round(3).to_string())

_k = sorted(TEXTS)[0]
_msgs = sanitize_messages(CTX[_k] + [{"role": "assistant", "content": TEXTS[_k]}])
_clean, _ = capture(_msgs, HOOK_LAYERS)
with steering(model, STEER_LAYER, REG_MATCHED, DOSE):
    _hooked, _ = capture(_msgs, HOOK_LAYERS)
print(f"\nscenario: {_k}")
for L in HOOK_LAYERS:
    print(f"layer {L}")
    print(f"  hook off      : {top_tokens(jlens_read(_clean[L], L))}")
    print(f"  register hook : {top_tokens(jlens_read(_hooked[L], L))}")

|axis| 1069.7 | |REG_MATCHED| 1069.7

REGISTER arm:
      kl_clean_to_hooked        top50_jaccard      shift_along_injection       
                    mean    std          mean  std                  mean    std
layer                                                                          
27                26.508  2.098           0.0  0.0                 0.758  0.044

AXIS arm, for comparison:
      kl_clean_to_hooked        top50_jaccard        shift_along_injection       
                    mean    std          mean    std                  mean    std
layer                                                                            
27                 5.291  1.545          0.06  0.034                  0.45  0.011

scenario: blackmail_explicit-america_none
layer 27
  hook off      : [' prioritization', ' proactively', ' proactive', ' prioritizing', ' leveraging', ' strategic', ' prioritized', ' alignment', ' operational', ' prioritize', ' assessment', ' priorities']
  register hook 